# Data Preprocessing — Image Captioning Project (Flickr8k / Flickr30k)
### Neural Networks and Deep Learning Course — Dr. Abdous

This notebook performs **only the preprocessing stage** (no model training) and produces a set of compressed, ready-to-use files that the subsequent notebooks (Part 1: LSTM, Part 2: Transformer Decoder, Part 3: Prompt-Guided) will load directly.

## Summary of Similar Projects Reviewed (Online)

Before writing the code, dozens of similar implementations (Kaggle Notebooks, GitHub repositories, and papers) for Image Captioning on Flickr8k/Flickr30k were examined. Common and frequently recurring points:

1. **Image Encoder:** Almost all use a CNN pre-trained on ImageNet (most common: `VGG16`, `InceptionV3`, `ResNet50`; newer works use `ViT`). The final Fully-Connected layers are removed and the feature map or pooled vector is used.
2. **Key trick to avoid OOM / Time-Limit:** Virtually every successful implementation (including the well-known Jason Brownlee tutorials that form the basis of most Flickr8k guides) extracts CNN features **once in advance and stores them on disk** so that raw images do not have to pass through the CNN on every epoch. This both multiplies training speed and reduces memory consumption.
3. **Text Decoder:** The baseline version uses an `LSTM` (with/without Attention); more advanced versions replace the decoder with a `Transformer` (Multi-Head Self-Attention + Cross-Attention + Positional Encoding) — exactly the architecture required in Part 2 of the project.
4. **Controllable / Prompt-Guided Captions (Part 3):** In the research literature this is known as *Controllable / Conditional Image Captioning*; the most common method, also stated in the project document, is **concatenation** of the Prompt embedding vector with the image features or inserting it as an initial token in the decoder input sequence.
5. **Data Split:** The split must be performed on *image IDs*, not on captions, because each image has 5 captions; splitting by caption would allow the same image to appear in both Train and Test (data leakage). This mistake appears in several beginner tutorials and is deliberately avoided here.
6. **Practical Kaggle constraints** (according to official documentation and discussions):
   - Notebook with GPU: approximately **16 GB RAM** and **13–16 GB VRAM** (depending on GPU type: P100 or T4×2), **4 CPU cores**.
   - Session time limit: **9–12 hours**.
   - Output folder `/kaggle/working` has a size ceiling near **20 GB** and a file-count ceiling near **500 files**; therefore storing thousands of small files (e.g. one file per image) is a common and dangerous mistake.
   - Cells that produce no output for a long time can cause a kernel timeout; long loops must therefore print progress periodically (heartbeat / `tqdm`).

## Strategy of This Notebook to Prevent OOM and Time-Limit

| Risk | Solution adopted in this notebook |
|---|---|
| Loading all images into RAM | Images are never kept entirely in memory; all processing is streaming and batch-wise on disk |
| Passing raw images through the CNN every training epoch | Image features are extracted once in this notebook with a frozen encoder and stored in a single compressed `HDF5` file using `float16` precision |
| Excessively large vocabulary | Rare words are removed (`MIN_WORD_FREQ`) and a hard ceiling is placed on vocabulary size — the Embedding layer and output Softmax depend directly on vocabulary size |
| Outlier caption lengths and excessive padding | Maximum length is set from the 95th percentile of the length distribution rather than the absolute maximum |
| Thousands of small files in the output | All artifacts are aggregated into a limited number of files (`vocab.json`, `captions.npz`, `splits.json`, `features.h5`, `manifest.json`) |
| Data leakage between Train/Val/Test | Split is performed by image ID with a fixed seed |
| Kernel crash due to long-running output with no log | Periodic progress printing (heartbeat) in all heavy loops |

> **Important note:** Extracting features with a frozen encoder means that in the training notebooks the CNN is no longer trained (not fine-tuned) — exactly what the project text permits and recommends. If you decide to fine-tune the encoder, skip the “feature cache” section and instead use the Resize+Normalize transform defined in this notebook on-the-fly in the training notebook (with a smaller batch size and mixed precision).


In [ ]:
# Install / check required packages (most are already present in the Kaggle image)
import importlib, subprocess, sys

REQUIRED = ["h5py", "tqdm", "torch", "torchvision", "pandas", "numpy", "PIL", "matplotlib", "kagglehub"]
missing = []
for pkg in REQUIRED:
    name = "PIL" if pkg == "PIL" else pkg
    try:
        importlib.import_module(name if name != "PIL" else "PIL")
    except ImportError:
        missing.append("Pillow" if pkg == "PIL" else pkg)

if missing:
    print("Installing missing packages:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
else:
    print("All required packages are available.")


In [ ]:
import os, re, json, gc, time, random, string, glob, shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as tvm

import h5py
import psutil

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def report_resources():
    vm = psutil.virtual_memory()
    disk = shutil.disk_usage("/kaggle/working" if os.path.exists("/kaggle/working") else ".")
    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"RAM total: {vm.total/1e9:.1f} GB | available: {vm.available/1e9:.1f} GB")
    print(f"Disk free (working dir): {disk.free/1e9:.1f} GB")

report_resources()


## 1. Configuration (Config)

All important preprocessing parameters are collected in one place. To switch between `Flickr8k` and `Flickr30k` only change `DATASET`.
You no longer need to manually attach the dataset with **Add Input**: the next section downloads it automatically (via `kagglehub`) if it is not already present. The Kaggle slug used for each option is:
- Flickr8k → dataset `adityajn105/flickr8k` (~1 GB, fast)
- Flickr30k → dataset `hsankesara/flickr-image-dataset` (~9 GB, slower to download)


In [ ]:
class CFG:
    # ---- Dataset selection ----
    DATASET = "flickr8k"          # or "flickr30k"
    SEED = 42

    # ---- Image ----
    IMG_SIZE = 224
    ENCODER = "resnet50"          # "resnet50" or "vit_b16"

    # ---- Text ----
    MIN_WORD_FREQ = 5             # words occurring less often than this are discarded (controls vocabulary size)
    MAX_VOCAB_SIZE = 10000        # hard ceiling on vocabulary size (including special tokens)
    LEN_PERCENTILE = 95           # maximum caption length is taken from this percentile
    HARD_MAX_LEN = 35             # absolute maximum caption length (including <start>/<end>)

    # ---- Data split ----
    TRAIN_RATIO = 0.80
    VAL_RATIO = 0.10
    TEST_RATIO = 0.10             # must sum to 1

    # ---- Feature extraction & caching ----
    EXTRACT_FEATURES = True       # strongly recommended: True (avoids OOM / time-limit during training)
    FEATURE_MODE = "spatial"      # "spatial" (49x2048 or 197x768) or "pooled" (2048 or 768)
    FEATURE_DTYPE = "float16"     # half the size of float32
    FEATURE_BATCH_SIZE = 64
    FEATURE_NUM_WORKERS = 2
    MAX_OUTPUT_GB_WARNING = 15.0  # warn if estimated cache size exceeds this value

    # ---- Paths ----
    OUTPUT_DIR = "/kaggle/working/preprocessed"

set_seed(CFG.SEED)
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
assert abs(CFG.TRAIN_RATIO + CFG.VAL_RATIO + CFG.TEST_RATIO - 1.0) < 1e-6
print(json.dumps({k: v for k, v in vars(CFG).items() if not k.startswith("_")}, indent=2, ensure_ascii=False))


## 2. Automatic Dataset Download + Locating and Loading the Data

This section first checks whether the dataset is already available under `/kaggle/input` (e.g. attached manually via *Add Input*). If it is not found, the dataset is **downloaded automatically** using `kagglehub` (Kaggle's official client library), so the notebook is fully self-contained and does not require any manual step.

- On Kaggle Notebooks, `kagglehub` authenticates automatically — just make sure **Internet** is turned on in the notebook's settings panel.
- Outside Kaggle (e.g. Colab, a local machine), set the `KAGGLE_USERNAME` and `KAGGLE_KEY` environment variables (from Kaggle -> Settings -> Create New API Token), or place a `kaggle.json` file at `~/.kaggle/kaggle.json`.
- The download is cached by `kagglehub` outside `/kaggle/working`, so it does **not** count against the `/kaggle/working` output quota, and re-running the notebook will reuse the cache instead of downloading again.

Three common caption-file formats are supported once the data is located:

1. CSV with header `image,caption` (common Flickr8k format on Kaggle)
2. CSV/text delimited by `|` such as `image_name| comment_number| comment` (common Flickr30k format)
3. Original format `Flickr8k.token.txt` with structure `image.jpg#0<TAB>caption`


In [ ]:
def find_images_dir(root="/kaggle/input"):
    """Find the directory that contains the largest number of jpg/png files."""
    best_dir, best_count = None, 0
    for dirpath, _, filenames in os.walk(root):
        n = sum(1 for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png")))
        if n > best_count:
            best_count, best_dir = n, dirpath
    return best_dir, best_count


def find_caption_file(root="/kaggle/input"):
    """Find the most likely caption file (csv/txt/token) by name and size."""
    candidates = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            fl = f.lower()
            if fl.endswith((".csv", ".txt", ".token")) and (
                "caption" in fl or "token" in fl or "result" in fl
            ):
                candidates.append(os.path.join(dirpath, f))
    if not candidates:
        return None
    # The largest file is usually the main caption file (not a readme, etc.)
    return max(candidates, key=lambda p: os.path.getsize(p))


def parse_captions(path):
    """Try to read the caption file in several common formats and return a DataFrame with columns image, caption."""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        head = [next(f) for _ in range(3)]

    # Format 1: image,caption
    if "," in head[0] and "image" in head[0].lower():
        rows = []
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            next(f)  # header
            for line in f:
                line = line.rstrip("\n")
                if not line:
                    continue
                img, cap = line.split(",", 1)
                rows.append((img.strip(), cap.strip()))
        return pd.DataFrame(rows, columns=["image", "caption"])

    # Format 2: image_name| comment_number| comment
    if "|" in head[0]:
        df = pd.read_csv(path, sep="|", engine="python", on_bad_lines="skip")
        df.columns = [c.strip().lower() for c in df.columns]
        img_col = [c for c in df.columns if "image" in c][0]
        cap_col = [c for c in df.columns if "comment" in c or "caption" in c][0]
        out = df[[img_col, cap_col]].copy()
        out.columns = ["image", "caption"]
        out["image"] = out["image"].astype(str).str.strip()
        out["caption"] = out["caption"].astype(str).str.strip()
        return out

    # Format 3: image.jpg#0<TAB>caption
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line or "\t" not in line:
                continue
            img_part, cap = line.split("\t", 1)
            img = img_part.split("#")[0].strip()
            rows.append((img, cap.strip()))
    return pd.DataFrame(rows, columns=["image", "caption"])


In [ ]:
import kagglehub

# Kaggle dataset slugs used for automatic download when the data is not already present
DATASET_SLUGS = {
    "flickr8k": "adityajn105/flickr8k",
    "flickr30k": "hsankesara/flickr-image-dataset",
}

MIN_IMAGES_TO_ACCEPT_EXISTING = 100  # heuristic: treat /kaggle/input as "already has the dataset" above this count


def get_dataset_root():
    """Return a directory that contains the dataset, downloading it automatically if needed."""
    # 1) Prefer a dataset already attached via "Add Input" (or already downloaded in a previous run)
    if os.path.exists("/kaggle/input"):
        existing_dir, existing_count = find_images_dir("/kaggle/input")
        if existing_dir is not None and existing_count > MIN_IMAGES_TO_ACCEPT_EXISTING:
            print(f"Found an existing dataset under /kaggle/input: {existing_dir} "
                  f"({existing_count} images). Skipping download.")
            return "/kaggle/input"

    # 2) Otherwise, download it automatically via kagglehub
    slug = DATASET_SLUGS[CFG.DATASET]
    print(f"No local dataset found. Downloading '{slug}' via kagglehub (this may take a while)...")
    t0 = time.time()
    try:
        download_path = kagglehub.dataset_download(slug)
    except Exception as e:
        raise RuntimeError(
            "Automatic dataset download failed. If this notebook is running on Kaggle, make sure "
            "'Internet' is turned ON in the notebook's settings panel. If it is running outside "
            "Kaggle, set the KAGGLE_USERNAME and KAGGLE_KEY environment variables (create an API "
            "token from your Kaggle account -> Settings -> Create New API Token), or place a "
            "kaggle.json file at ~/.kaggle/kaggle.json, then re-run this cell."
        ) from e
    print(f"Dataset downloaded to: {download_path} (took {(time.time() - t0) / 60:.1f} min)")
    return download_path


DATASET_ROOT = get_dataset_root()


In [ ]:
images_dir, n_images_found = find_images_dir(DATASET_ROOT)
caption_path = find_caption_file(DATASET_ROOT)
print("Located image directory:", images_dir, "| number of images:", n_images_found)
print("Located caption file:", caption_path)

assert images_dir is not None, "Image directory not found even after automatic download; check CFG.DATASET / DATASET_SLUGS."
assert caption_path is not None, "Caption file not found even after automatic download; check CFG.DATASET / DATASET_SLUGS."

df = parse_captions(caption_path)
df = df.dropna().drop_duplicates()
df["image"] = df["image"].apply(lambda x: os.path.basename(x))
print(df.shape)
df.head()


In [ ]:
# Build the full path of every image and keep the mapping image -> full path (without loading the image itself)
image_path_map = {}
for dirpath, _, filenames in os.walk(images_dir):
    for f in filenames:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            image_path_map[f] = os.path.join(dirpath, f)

df["path"] = df["image"].map(image_path_map)
before = len(df)
df = df.dropna(subset=["path"]).reset_index(drop=True)
print(f"Number of rows removed because the corresponding image file was missing: {before - len(df)}")
print(f"Total number of captions: {len(df)} | number of unique images: {df['image'].nunique()}")


## 3. Image Integrity Check

Corrupted / incomplete files are removed before any further processing. This is done in a **streaming** fashion (one image at a time, never held in memory) with periodic progress reports so that RAM is not exhausted and a long-running cell does not cause a timeout.


In [ ]:
def verify_images(paths, log_every=1000):
    bad = set()
    t0 = time.time()
    for i, p in enumerate(tqdm(paths, desc="Verifying images")):
        try:
            with Image.open(p) as im:
                im.verify()
        except Exception:
            bad.add(p)
        if (i + 1) % log_every == 0:
            gc.collect()
            print(f"  [{i+1}/{len(paths)}] checked | corrupted so far: {len(bad)} | "
                  f"elapsed: {time.time()-t0:.1f}s", flush=True)
    return bad

unique_paths = df["path"].unique().tolist()
bad_paths = verify_images(unique_paths)
print(f"Number of corrupted / incomplete files: {len(bad_paths)}")

if bad_paths:
    df = df[~df["path"].isin(bad_paths)].reset_index(drop=True)
    print(f"Number of rows remaining after removing corrupted images: {len(df)}")


## 4. Text Preprocessing (Captions)

Steps required by the project document: lower-casing, punctuation removal, tokenization, vocabulary construction, and definition of special tokens.


In [ ]:
PUNCT_RE = re.compile(f"[{re.escape(string.punctuation)}]")
MULTISPACE_RE = re.compile(r"\s+")

def clean_caption(text: str) -> str:
    text = text.lower().strip()
    text = PUNCT_RE.sub(" ", text)          # remove punctuation
    text = re.sub(r"\d+", " ", text)        # remove numbers (optional but common)
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text

tqdm.pandas(desc="Cleaning captions")
df["clean_caption"] = df["caption"].progress_apply(clean_caption)
df = df[df["clean_caption"].str.len() > 0].reset_index(drop=True)

df[["image", "caption", "clean_caption"]].head()


### Building the Vocabulary

`MIN_WORD_FREQ` and `MAX_VOCAB_SIZE` are deliberately restrictive: vocabulary size directly determines the size of the Embedding layer and the output Softmax of the decoder and is one of the most common sources of OOM in this kind of project.


In [ ]:
SPECIAL_TOKENS = ["<pad>", "<start>", "<end>", "<unk>"]
PAD_ID, START_ID, END_ID, UNK_ID = 0, 1, 2, 3

word_counter = Counter()
for cap in df["clean_caption"]:
    word_counter.update(cap.split())

# Filter by minimum frequency, then sort by frequency to apply the vocabulary-size ceiling
kept = [w for w, c in word_counter.items() if c >= CFG.MIN_WORD_FREQ]
kept.sort(key=lambda w: -word_counter[w])
budget = CFG.MAX_VOCAB_SIZE - len(SPECIAL_TOKENS)
kept = kept[:budget]

idx2word = SPECIAL_TOKENS + kept
word2idx = {w: i for i, w in enumerate(idx2word)}
VOCAB_SIZE = len(idx2word)

total_tokens = sum(word_counter.values())
unk_tokens = sum(c for w, c in word_counter.items() if w not in word2idx)
print(f"Final vocabulary size: {VOCAB_SIZE}")
print(f"Percentage of tokens that become <unk>: {100*unk_tokens/total_tokens:.2f}%")

with open(os.path.join(CFG.OUTPUT_DIR, "vocab.json"), "w", encoding="utf-8") as f:
    json.dump({"word2idx": word2idx, "idx2word": idx2word,
               "special_tokens": {"pad": PAD_ID, "start": START_ID, "end": END_ID, "unk": UNK_ID}},
              f, ensure_ascii=False, indent=1)


### Determining Maximum Length and Encoding Captions

Instead of using the longest caption present (which may be an outlier and cause a large amount of unnecessary padding), the 95th percentile of the length distribution is used.


In [ ]:
token_lengths = df["clean_caption"].apply(lambda s: len(s.split()) + 2)  # +2 for <start>/<end>
p95 = int(np.percentile(token_lengths, CFG.LEN_PERCENTILE))
MAX_LEN = min(max(p95, 8), CFG.HARD_MAX_LEN)
print(f"Token length at percentile {CFG.LEN_PERCENTILE}: {p95} | final MAX_LEN: {MAX_LEN}")

trunc_ratio = (token_lengths > MAX_LEN).mean()
print(f"Percentage of captions that are truncated because of MAX_LEN: {100*trunc_ratio:.2f}%")


def encode_caption(clean_text: str, max_len: int = MAX_LEN):
    tokens = clean_text.split()
    ids = [START_ID] + [word2idx.get(t, UNK_ID) for t in tokens] + [END_ID]
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids = ids + [PAD_ID] * (max_len - len(ids))
    return ids

tqdm.pandas(desc="Encoding captions")
encoded = np.stack(df["clean_caption"].progress_apply(encode_caption).to_list()).astype(np.int32)
print("Shape of encoded caption array:", encoded.shape)


## 5. Data Split (Train / Val / Test)

**Critical point:** The split is performed on `image`, not on caption rows; otherwise captions of the same image could end up in both Train and Test (data leakage).


In [ ]:
unique_images = df["image"].unique()
rng = np.random.RandomState(CFG.SEED)
shuffled = rng.permutation(unique_images)

n = len(shuffled)
n_train = int(n * CFG.TRAIN_RATIO)
n_val = int(n * CFG.VAL_RATIO)

train_imgs = set(shuffled[:n_train])
val_imgs = set(shuffled[n_train:n_train + n_val])
test_imgs = set(shuffled[n_train + n_val:])

def assign_split(img):
    if img in train_imgs:
        return "train"
    if img in val_imgs:
        return "val"
    return "test"

df["split"] = df["image"].apply(assign_split)
print(df["split"].value_counts())
print("Number of unique images per split:",
      {s: len(v) for s, v in [("train", train_imgs), ("val", val_imgs), ("test", test_imgs)]})

with open(os.path.join(CFG.OUTPUT_DIR, "splits.json"), "w", encoding="utf-8") as f:
    json.dump({"train": sorted(train_imgs), "val": sorted(val_imgs), "test": sorted(test_imgs)}, f, indent=1)

# Store encoded captions + metadata in a single compressed file (instead of thousands of separate files)
np.savez_compressed(
    os.path.join(CFG.OUTPUT_DIR, "captions.npz"),
    image=df["image"].to_numpy(),
    split=df["split"].to_numpy(),
    raw_caption=df["caption"].to_numpy(),
    clean_caption=df["clean_caption"].to_numpy(),
    token_ids=encoded,
)
print("captions.npz saved.")


## 6. Image Transform Definition (Resize + Normalize)

Size and normalization statistics are chosen to match the selected pre-trained encoder (`CFG.ENCODER`). This transform is used both for the feature-caching stage in this notebook and (if the fine-tune path is chosen) on-the-fly in the training notebook.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Resize(256) + CenterCrop(224) is exactly the standard preprocessing of ImageNet pre-trained models
image_transform = T.Compose([
    T.Resize(int(CFG.IMG_SIZE * 256 / 224)),
    T.CenterCrop(CFG.IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

transform_config = {
    "img_size": CFG.IMG_SIZE,
    "resize_to": int(CFG.IMG_SIZE * 256 / 224),
    "center_crop": CFG.IMG_SIZE,
    "mean": IMAGENET_MEAN,
    "std": IMAGENET_STD,
    "encoder": CFG.ENCODER,
}
with open(os.path.join(CFG.OUTPUT_DIR, "transform_config.json"), "w") as f:
    json.dump(transform_config, f, indent=1)

print(image_transform)


## 7. Caching Image Features with a Frozen Encoder (Key Anti-OOM / Time-Limit Section)

- Images pass once through a **frozen** CNN/ViT (no gradients, no training).
- The output of each image (spatial feature map, e.g. `49×2048` for ResNet50 or `197×768` for ViT-B/16) is written with `float16` precision directly into a single **HDF5** file on disk; the whole set is never kept in RAM.
- In the training notebooks (Parts 1, 2, 3) there is no longer any need to pass a raw image through the CNN — only the feature vector / map is read from the HDF5 file. This drastically reduces the time of every epoch and almost eliminates GPU-memory OOM risk (because the CNN computational graph is never built).
- The `spatial` format (spatial map, not merely a pooled vector) is deliberately chosen as the default because it is useful for both an LSTM decoder and a Transformer decoder (Part 2, including Cross-Attention) as well as for attention-map analysis.

> If you want to fine-tune the CNN, do not run this section (`CFG.EXTRACT_FEATURES = False`) and in the training notebook use the `image_transform` defined above on-the-fly (with a smaller batch size, `torch.cuda.amp` and, if needed, gradient accumulation).


In [ ]:
def build_frozen_encoder(name: str):
    if name == "resnet50":
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
        backbone = nn.Sequential(*list(m.children())[:-2])  # output: (B, 2048, 7, 7)
        num_tokens, feat_dim = 49, 2048

        def forward(x):
            f = backbone(x)                          # (B, 2048, 7, 7)
            f = f.flatten(2).transpose(1, 2)          # (B, 49, 2048)
            return f

        return backbone, forward, num_tokens, feat_dim

    elif name == "vit_b16":
        m = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1)
        m.heads = nn.Identity()
        num_tokens, feat_dim = 197, 768  # 196 patch + 1 cls

        def forward(x):
            x = m._process_input(x)
            n = x.shape[0]
            cls = m.class_token.expand(n, -1, -1)
            x = torch.cat([cls, x], dim=1)
            x = x + m.encoder.pos_embedding
            x = m.encoder.dropout(x)
            x = m.encoder.layers(x)
            x = m.encoder.ln(x)
            return x                                  # (B, 197, 768)

        return m, forward, num_tokens, feat_dim

    else:
        raise ValueError(f"Unknown encoder: {name}")


class ImageOnlyDataset(Dataset):
    """Reads only the image and applies Resize+Normalize; does not deal with captions (for feature extraction)."""
    def __init__(self, image_names, path_map, transform):
        self.image_names = image_names
        self.path_map = path_map
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        name = self.image_names[idx]
        path = self.path_map[name]
        try:
            with Image.open(path) as im:
                im = im.convert("RGB")
                tensor = self.transform(im)
        except Exception:
            tensor = torch.zeros(3, CFG.IMG_SIZE, CFG.IMG_SIZE)
        return name, tensor


In [ ]:
if CFG.EXTRACT_FEATURES:
    unique_images_all = df["image"].unique().tolist()

    _, _, num_tokens, feat_dim = build_frozen_encoder(CFG.ENCODER)
    if CFG.FEATURE_MODE == "pooled":
        num_tokens = 1
    bytes_per_elem = 2 if CFG.FEATURE_DTYPE == "float16" else 4
    est_bytes = len(unique_images_all) * num_tokens * feat_dim * bytes_per_elem
    est_gb = est_bytes / 1e9
    print(f"Estimated size of the feature-cache file: {est_gb:.2f} GB "
          f"({len(unique_images_all)} images × {num_tokens} tokens × {feat_dim} dims × {bytes_per_elem} bytes)")

    if est_gb > CFG.MAX_OUTPUT_GB_WARNING:
        print("Warning: estimated size exceeds the configured threshold! "
              "Suggestion: FEATURE_MODE='pooled' or FEATURE_DTYPE='float16' or reduce the number of images.")
else:
    print("EXTRACT_FEATURES=False; this section is skipped (fine-tune path selected).")


In [ ]:
def extract_and_cache_features():
    backbone, forward_fn, num_tokens, feat_dim = build_frozen_encoder(CFG.ENCODER)
    backbone = backbone.to(DEVICE).eval()
    for p in backbone.parameters():
        p.requires_grad_(False)

    unique_images_all = df["image"].unique().tolist()
    ds = ImageOnlyDataset(unique_images_all, image_path_map, image_transform)
    loader = DataLoader(ds, batch_size=CFG.FEATURE_BATCH_SIZE, shuffle=False,
                         num_workers=CFG.FEATURE_NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

    n = len(unique_images_all)
    out_tokens = 1 if CFG.FEATURE_MODE == "pooled" else num_tokens
    h5_path = os.path.join(CFG.OUTPUT_DIR, "features.h5")

    np_dtype = np.float16 if CFG.FEATURE_DTYPE == "float16" else np.float32
    id2idx = {}

    t0 = time.time()
    with h5py.File(h5_path, "w") as hf:
        feat_ds = hf.create_dataset(
            "features", shape=(n, out_tokens, feat_dim), dtype=np_dtype,
            chunks=(1, out_tokens, feat_dim), compression="gzip", compression_opts=4,
        )
        row = 0
        use_amp = DEVICE.type == "cuda"
        for batch_idx, (names, imgs) in enumerate(tqdm(loader, desc=f"Extracting {CFG.ENCODER} features")):
            imgs = imgs.to(DEVICE, non_blocking=True)
            with torch.no_grad(), torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                feats = forward_fn(imgs)               # (B, num_tokens, feat_dim)
                if CFG.FEATURE_MODE == "pooled":
                    feats = feats.mean(dim=1, keepdim=True)  # (B, 1, feat_dim)
            feats = feats.float().cpu().numpy().astype(np_dtype)

            b = feats.shape[0]
            feat_ds[row:row + b] = feats
            for name in names:
                id2idx[name] = row
                row += 1

            del imgs, feats
            if batch_idx % 20 == 0:
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()
                gc.collect()
                elapsed = time.time() - t0
                done = row / n
                eta = elapsed / max(done, 1e-6) * (1 - done)
                print(f"  [{row}/{n}] {100*done:.1f}% | elapsed: {elapsed/60:.1f} min | "
                      f"estimated remaining: {eta/60:.1f} min", flush=True)

    with open(os.path.join(CFG.OUTPUT_DIR, "feature_index.json"), "w") as f:
        json.dump({
            "id2idx": id2idx, "num_tokens": out_tokens, "feat_dim": feat_dim,
            "dtype": CFG.FEATURE_DTYPE, "encoder": CFG.ENCODER, "mode": CFG.FEATURE_MODE,
        }, f)

    size_gb = os.path.getsize(h5_path) / 1e9
    print(f"Feature extraction finished. Final file: {h5_path} ({size_gb:.2f} GB) | "
          f"total time: {(time.time()-t0)/60:.1f} min")
    return h5_path


if CFG.EXTRACT_FEATURES:
    features_h5_path = extract_and_cache_features()
else:
    features_h5_path = None


## 8. Display a 5×5 Grid of Training Samples

Randomly show 25 samples from the Train set together with one corresponding caption, purely to verify the correctness of the image↔caption mapping (images are read only for display and are not stored).


In [ ]:
import textwrap

train_df = df[df["split"] == "train"]
sample_rows = train_df.sample(n=25, random_state=CFG.SEED).reset_index(drop=True)

fig, axes = plt.subplots(5, 5, figsize=(18, 20))
for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    with Image.open(row["path"]) as im:
        im = im.convert("RGB")
        ax.imshow(im)
    ax.axis("off")
    wrapped = textwrap.fill(row["clean_caption"], width=28)
    ax.set_title(wrapped, fontsize=8)

plt.tight_layout()
grid_path = os.path.join(CFG.OUTPUT_DIR, "train_batch_grid_5x5.png")
plt.savefig(grid_path, dpi=120, bbox_inches="tight")
plt.show()
print("Grid saved at:", grid_path)


## 9. Saving the Final Manifest and Checking Kaggle File/Size Limits

A single `manifest.json` file summarizes all paths, settings and statistics for the training notebooks. The size and number of generated files are also compared against the known Kaggle limits (~20 GB, ~500 files for `/kaggle/working`).


In [ ]:
out_files = list(Path(CFG.OUTPUT_DIR).rglob("*"))
out_files = [p for p in out_files if p.is_file()]
total_size_gb = sum(p.stat().st_size for p in out_files) / 1e9

print(f"Number of generated files: {len(out_files)} (Kaggle ceiling ≈ 500)")
print(f"Total output size: {total_size_gb:.3f} GB (Kaggle ceiling ≈ 20 GB)")
for p in sorted(out_files):
    print(f"  - {p.relative_to(CFG.OUTPUT_DIR)}: {p.stat().st_size/1e6:.2f} MB")

if len(out_files) > 450:
    print("Warning: number of files is approaching the Kaggle ceiling.")
if total_size_gb > 18:
    print("Warning: output size is approaching the Kaggle ceiling.")

manifest = {
    "dataset": CFG.DATASET,
    "seed": CFG.SEED,
    "num_captions": int(len(df)),
    "num_unique_images": int(df["image"].nunique()),
    "vocab_size": int(VOCAB_SIZE),
    "max_caption_len": int(MAX_LEN),
    "split_sizes_images": {"train": len(train_imgs), "val": len(val_imgs), "test": len(test_imgs)},
    "image_transform": transform_config,
    "features": {
        "enabled": CFG.EXTRACT_FEATURES,
        "path": features_h5_path,
        "mode": CFG.FEATURE_MODE,
        "encoder": CFG.ENCODER,
        "dtype": CFG.FEATURE_DTYPE,
    },
    "files": {
        "vocab": "vocab.json",
        "captions": "captions.npz",
        "splits": "splits.json",
        "transform_config": "transform_config.json",
        "feature_index": "feature_index.json" if CFG.EXTRACT_FEATURES else None,
        "features_h5": "features.h5" if CFG.EXTRACT_FEATURES else None,
        "sample_grid": "train_batch_grid_5x5.png",
    },
}
with open(os.path.join(CFG.OUTPUT_DIR, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=1)

print("\nmanifest.json saved. Preprocessing complete.")


## Summary and Checklist Before Moving to the Training Notebook

- [x] Images or features were never kept entirely in RAM (streaming / batch processing).
- [x] CNN/ViT features were extracted once with a frozen encoder and cached in `features.h5` (single file, `float16`) → during training there is no need to pass a raw image through the CNN.
- [x] Vocabulary size was controlled with `MIN_WORD_FREQ` and `MAX_VOCAB_SIZE` → smaller and faster Embedding / Softmax layers.
- [x] Caption length was set from the 95th percentile (not the absolute maximum) → less unnecessary padding.
- [x] Train/Val/Test split was performed by image → no data leakage.
- [x] All artifacts were stored in a few aggregated files, not thousands of small files → compatible with Kaggle file/size limits.
- [x] Heavy loops (image verification, feature extraction) printed progress logs periodically → prevention of kernel timeout.

**Next step:** In the Part-1 training notebook (Baseline CNN+LSTM) it is sufficient to load these 5 files from `/kaggle/working/preprocessed/` (or from a Kaggle Dataset created from this folder): `manifest.json`, `vocab.json`, `captions.npz`, `splits.json`, `features.h5`. There is no longer any need for the raw images or the CNN itself inside the training loop, unless the fine-tune path has been chosen.

> **Practical recommendation:** After a successful run of this notebook, publish the `preprocessed/` output folder as a new Kaggle Dataset (Output → New Dataset). This lets the training notebooks simply Add Input the folder and avoids re-running the (GPU-time-consuming) preprocessing every time.
